# 05 — Recurrent Models

Purpose: Evaluate Vanilla LSTM, Bidirectional LSTM, GRU, and Attention LSTM with validation-battery early stopping.

Leakage rule: current-cycle full discharged capacity and SOH are labels only.
Preprocessing is fitted on training batteries, validation batteries control
selection and early stopping, and test batteries are scored exactly once.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v3"
RESULTS = ARTIFACT_ROOT / "results"
FIGURES = ARTIFACT_ROOT / "figures"
FEATURES = ARTIFACT_ROOT / "features"
for path in (RESULTS, FIGURES, FEATURES):
    path.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")


Project root: E:\VIT\aiBatteryLifecycle\application


In [2]:
import numpy as np, pandas as pd
from src.experiments.deep import run_grouped_sequence_benchmark
data = np.load(FEATURES / "nasa" / "sequences.npz")
index = pd.read_csv(FEATURES / "nasa" / "sequence_index.csv")
models = ("vanilla_lstm", "bidirectional_lstm", "gru", "attention_lstm")
metric_path = RESULTS / "nasa_recurrent_fold_metrics.csv"
prediction_path = RESULTS / "nasa_recurrent_predictions.csv"
if not metric_path.exists() or not prediction_path.exists():
    metrics, predictions = run_grouped_sequence_benchmark(data["X"], index, dataset_name="NASA", model_ids=models)
    metrics.to_csv(metric_path, index=False)
    predictions.to_csv(prediction_path, index=False)
else:
    metrics = pd.read_csv(metric_path)
display(metrics.groupby("model")[["mae", "rmse", "r2", "within_5pp"]].mean().sort_values("mae"))

,mae,rmse,r2,within_5pp
model,,,,
bidirectional_lstm,6.930352,9.634718,-0.272079,0.534427
vanilla_lstm,7.493510,9.591145,-0.874047,0.472635
gru,7.618810,9.640196,-0.639237,0.446301
attention_lstm,8.171034,10.762947,-1.363875,0.480025
